## Integrated Project: Validating our Dataa

#### in this Code Challenge, we're diving into the agricultural dataset again to continue to validate our data. Before we do that, we're pausing to build a data pipeline that will ingest and clran our data with the press of a button, cleaning up our code significantly. Once that's ready, we'll complete our data validation

### Instructions
#### - **Do not add or remove cells in this notebook. Do not edit or remove the ### START FUNCTION or ### END FUNCTION comments. Do not add any code outside of the functions you are required to edit. Doing any of this will lead to a mark of 0%!**
#### - Answer the question according to the specifications provided.
#### - Use the given cell in each question to see if your function matches the expected outputs.
#### - Do not hard-code answers to the questions.
#### - The use of StackOverflow, Google, and other online tools is permitted. However, copying a fellow student's code is not permissible and is considered a breach of the Honour code. Doing this will result in a mark of 0%.

In [2]:
import re
import numpy as np
import pandas as pd
import os

In [14]:
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import seaborn as sns

# Create an engine for the database
engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')

sql_query = """
SELECT *
FROM geographic_features
LEFT JOIN weather_features USING (Field_ID)
LEFT JOIN soil_and_crop_features USING (Field_ID)
LEFT JOIN farm_management_features USING (Field_ID)
"""

# Create a connection object
with engine.connect() as connection:
    MD_agric_df = pd.read_sql_query(text(sql_query), connection)

DatabaseError: (sqlite3.DatabaseError) file is not a database
[SQL: 
SELECT *
FROM geographic_features
LEFT JOIN weather_features USING (Field_ID)
LEFT JOIN soil_and_crop_features USING (Field_ID)
LEFT JOIN farm_management_features USING (Field_ID)
]
(Background on this error at: https://sqlalche.me/e/14/4xp6)

In [10]:
def create_db_engine(db_path):
    engine = create_engine(db_path)
    return engine

def query_data(engine, sql_query):
    with engine.connect() as connection:
        df = pd.read_sql_query(text(sql_query), connection)
        return df

In [9]:
weather_station_df = pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv")
weather_stationi_mapping_df = pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv")

In [17]:
import logging
import pandas as pd

# Name our logger so we know that logs from this module come from the data_ingestion module
logger = logging.getLogger('data_ingestion')

# set a basic logging message up that prints out a timestamp, the name of our logger, and the message
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

def create_db_engine(db_path):
    try:
        engine = create_engine(db_path)
        # Test connection
        with engine.connect() as conn:
            pass
        # test if the dataase engine was created successfully
        logger.info("Database engine created successfully.")
        return engine  # Return the engine object if it all works well
    except ImportError:  # If we get an importError, inform the user SQLAlchemy is not installed
        logger.error("SQLAlchemy is required to use this function. Please install it first.")
    except Exception as e:  # If we fail to create an engine inform the user
        logger.error(f"Failed to create database engine. Error: {e}")
        raise e

def query_data(engine, sql_query):
    try:
        with engine.connect() as connection:
            df = pd.read_sql_query(text(sql_query), connection)
        if df.empty:
            # Log a message or handle the empty DataFrame scenario as needed
            msg = "The query returned an empty DataFrame."
            logger.error(msg)
            raise ValueError(msg)
        logger.info("Query executed successfully.")
        return df
    except ValueError as e:
        logger.error(f"SQL query failed. Error: {e}")
        raise e
    except Exception as e:
        logger.error(f"An error occurred while querying the database. Error: {e}")
        raise e

In [20]:
SQL_engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')

sql_query = """
SELECT *
FROM geographic_features
LEFT JOIN weather_features USING (Field_ID)
LEFT JOIN soil_and_crop_features USING (Field_ID)
LEFT JOIN farm_management_features USING (Field_ID)
WHERE Rainfall < 0
"""

# The last line won't ever be true, so no results will be returned
df = query_data(SQL_engine, sql_query)
df

2026-06-30 16:57:04,641 - data_ingestion - ERROR - The query returned an empty DataFrame.
2026-06-30 16:57:04,660 - data_ingestion - ERROR - SQL query failed. Error: The query returned an empty DataFrame.


ValueError: The query returned an empty DataFrame.

In [21]:
weather_data_URL = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv"
weather_mapping_data_URL = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"

def read_from_web_CSV(URL):
    try:
        df = pd.read_csv(URL)
        logger.info("CSV file read successfully from the web")
        return df
    except pd.errors.EmpyDataError as e:
        logger.error("The URL does not point to a valid CSV file. Please check the URL and try again.")
        raise e
    except Exception as e:
        logger.error(f"Failed to read CSV from the web. Error: {e}")
        raise e

weather_df = read_from_web_CSV(weather_data_URL)
weather_mapping_data = read_from_web_CSV(weather_mapping_data_URL)

2026-06-30 16:58:31,142 - data_ingestion - INFO - CSV file read successfully from the web
2026-06-30 16:58:32,359 - data_ingestion - INFO - CSV file read successfully from the web


In [32]:
%load_ext sql

In [38]:
%sql sqlite:///Maji_Ndogo_farm_survey_small.db

In [39]:
%%sql

SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;

Running query in 'sqlite:///Maji_Ndogo_farm_survey_small.db'

name
farm_management_features
geographic_features
soil_and_crop_features
weather_features


In [48]:
"""
This module implements logic to connect to a SQLite3 database and remote CSV
data sources
"""
from sqlalchemy import create_engine, text
import logging
import pandas as pd

# Name our logger so we know that logs from this module come from the data_ingestion module
logger = logging.getLogger('data_ingestion')
# Set a basic logging message up that prints out a timestamp, the name of our logger, and the message
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
db_path = 'sqlite:///Maji_Ndogo_farm_survey_small.db'

sql_query = """
SELECT *
FROM geographic_features
LEFT JOIN weather_features using (Field_ID)
LEFT JOIN soil_and_crop_features USING (Field_ID)
LEFT JOIN farm_management_features USING (Field_ID)
"""

weather_data_URL = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv"
weather_mapping_data_URL = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"

def create_db_engine(db_path):
    """
    Establishes a connection with the SQlite3 database provided
    in the db_path argument
    
    Args:
        db_path: The path to the database file
        
    Returns:
        The engine object to connect to the database if successful,
        an error message and a raised exception if unsuccessful
    """
    try:
        engine = create_engine(db_path)
        # Test connection
        with engine.connect() as conn:
            pass
        # Test if the dataase engine was created successfully
        logger.info("Database engine created successfully.")
        return engine  # Return the engine object if it all works well
    except ImportError:  # If we get an ImportError, inform the user SQLAlchemy is not installed
        logger.error("SQLAlchemy is required to use this function. Please install it first.")
        raise e
    except Exception as e:  # If we failt to create an engine inform the user
        logger.error(f"Failed to create database engine. Error: {e}")
        raise e

def query_data(engine, sql_query):
    """
    Connects to the database using the engine object returned by create_db_engine
    and executes the sql_query passed on the database through the engine object
    provided.
    
    Args:
        engine: The engine object returned by create_db_engine() when executed
                successfully
        sql_query: The query to execute on the SQlite3 database
    
    Returns:
        A Pandas DataFrame containing the data from the executed SQL query if
        executed successfully, a logged error message and a raised exception
        if unsuccessful
    """
    try:
        with engine.connect() as connection:
            df = pd.read_sql_query(text(sql_query), connection)
        if df.empty:
            # Log a message or handle the empty DataFrame scenario as needed
            msg = "The query returned an empty DataFrame."
            logger.error(msg)
            raise ValueError(msg)
        logger.info("Query executed successfully.")
        return df
    except ValueError as e:
        logger.error(f"SQL query failed: Error: {e}")
        raise e
    except Exception as e:
        logger.error(f"An error occurred while querying the database. Error: {e}")
        raise e

def read_from_web_CSV(URL):
    """
    Takes a remote URL, reads data from it and stores it in a Pandas Dataframe
    if successful but raises an error and an exception if unsuccessful
    
    Args:
        URL - The URL to the CSV files
    
    Returns:
        A Pandas DataFrame containing the CSV content if successful, an error
        and a raise exception otherwise
    """
    try:
        df = pd.read_csv(URL)
        logger.info("CVS file read successfully from the web.")
        return df
    except pd.errors.EmptyDataError as e:
        logger.error("The URL does not point to a valid CVS file. Please check the URL and try again.")
        raise e
    except Exception as e:
        logger.error(f"Failed to read CSV from the web. Error: {e}")
        raise e

In [49]:
# Testing module functions
field_df = query_data(create_db_engine(db_path), sql_query)
weather_df = read_from_web_CSV(weather_data_URL)
weather_mapping_df = read_from_web_CSV(weather_mapping_data_URL)

2026-06-30 20:47:20,171 - data_ingestion - INFO - Database engine created successfully.
2026-06-30 20:47:23,367 - data_ingestion - INFO - Query executed successfully.
2026-06-30 20:47:27,682 - data_ingestion - INFO - CVS file read successfully from the web.
2026-06-30 20:47:28,941 - data_ingestion - INFO - CVS file read successfully from the web.


In [50]:
field_test = field_df.shape
weather_test = weather_df.shape
weather_mapping_test = weather_mapping_df.shape
print(f"field_df: {field_test}, weather_df: {weather_test}, weather_mapping_df: {weather_mapping_test}")

field_df: (5654, 18), weather_df: (1843, 2), weather_mapping_df: (5654, 3)


In [53]:
# importing our new module
from data_ingestion import create_db_engine, query_data, read_from_web_CSV

# Checking if the function names are now associated with the module
print(create_db_engine.__module__)
print(query_data.__module__)
print(read_from_web_CSV.__module__)

data_ingestion
data_ingestion
data_ingestion


In [54]:
field_df = query_data(create_db_engine(db_path), sql_query)
weather_df = read_from_web_CSV(weather_data_URL)
weather_mapping_df = read_from_web_CSV(weather_mapping_data_URL)

field_test = field_df.shape
weather_test = weather_df.shape
weather_mapping_test = weather_mapping_df.shape
print(f"field_df: {field_test}, weather_df: {weather_test}, weather_mapping_df: {weather_mapping_test}")

2026-06-30 20:59:36,336 - data_ingestion - INFO - Database engine created successfully.
2026-06-30 20:59:36,499 - data_ingestion - INFO - Query executed successfully.
2026-06-30 20:59:37,382 - data_ingestion - INFO - CVS file read successfully from the web.
2026-06-30 20:59:38,154 - data_ingestion - INFO - CVS file read successfully from the web.


field_df: (5654, 18), weather_df: (1843, 2), weather_mapping_df: (5654, 3)


### Field data Processor

In [56]:
MD_agric_df = field_df.copy()

MD_agric_df.rename(columns={'Annual_yield': 'Crop_type_Temp', 'Crop_type': 'Annual_yield'}, inplace=True)
MD_agric_df.rename(columns={'Crop_type_Temp': 'Crop_type'}, inplace=True)
MD_agric_df['Elevation'] = MD_agric_df['Elevation'].abs()

# Correcting 'Crop_type' column
def correct_crop_type(crop):
    corrections = {
        'cassava1': 'cassava',
        'wheatn': 'wheat',
        'teaa': 'tea'
    }
    return corrections.get(crop, crop)  # Get the corrected crop type, or return the original if not in corrections

# Apply the correction function to the Crop_type column
MD_agric_df['Crop_type'] = MD_agric_df['Crop_type'].apply(correct_crop_type)

In [65]:
import pandas as pd
from data_ingestion import create_db_engine, query_data, read_from_web_CSV
import logging

class FieldDataProcessor:
    def __init__(self, logging_level="INFO"):  # When we instantiate this class, we can optionally specify what logs we want to see
        # initialising class with attributes we need. Refer to the code above to understand how each attribute relates to the code
        self.db_path = 'sqlite:///Maji_Ndogo_farm_survey_small.db'
        self.sql_query = """
            SELECT *
            FROM geographic_features
            LEFT JOIN weather_features USING (Field_ID)
            LEFT JOIN soil_and_crop_features USING (Field_ID)
            LEFT JOIN farm_management_features USING (Field_ID)
        """
        self.columns_to_rename = {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'}
        self.values_to_rename = {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
        self.weather_map_data = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"
        
        self.initialize_logging(logging_level)
        
        # We create empty objects to store the DataFrame and engine in
        self.df = None
        self.engine = None
    
    # This method enables logging in the class
    def initialize_logging(self, logging_level):
        """
        Sets up Logging for this instance of FieldDataProcessor
        """
        logger_name = __name__ + ".FieldDataProcessor"
        self.logger = logging.getLogger(logger_name)
        self.logger.propagate = False  # Prevents log messages from being propagated to the root logger
        
        # Set logging level
        if logging_level.upper() == "DEBUG":
            log_level = logging.DEBUG
        elif logging_level.upper() == "INFO":
            log_level = logging.INFO
        elif logging_level.upper() == "NONE":  # Option to disable logging
            self.logger.disabled = True
            return
        else:
            log_level = logging.INFO  # Default to INFO
        
        self.logger.setLevel(log_level)
        
        # Only add handler if not already added to avoid duplicate messages
        if not self.logger.handlers:
            ch = logging.StreamHandler()  # Create console handler
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            ch.setFormatter(formatter)
            self.logger.addHandler(ch)
        
        # Use self.logger.info(), self.logger.debug(), etc.
    
    # DataFrame methods
    def ingest_sql_data(self):
        # First we want to get the data from the SQL database
        pass
    
    def rename_columns(self):
        # Annual_yield and Crop_type must be swapped
        pass
    
    def apply_corrections(self):
        # Merge the weather station data to the main DataFrame
        pass
    
    def weather_station_mapping(self):
        # Merge the weather station data to the main DataFrame
        pass
    
    def process(self):
        # This process calls the correct methods and applies the changes, step by step. This is the method we will call, and it will call the other methods in order
        weather_map_df = self.weather_station_mapping()
        self.df = self.ingest_sql_data()
        self.df = self.rename_columns()
        self.df = self.apply_corrections()
        self.df = self.df.merge(weather_map_df, on='Field_ID', how='left')
        self.df = self.df.drop(columns="Unnamed: 0")

In [66]:
field_processor = FieldDataProcessor()
field_processor.process()

AttributeError: 'NoneType' object has no attribute 'merge'

In [67]:
field_df = field_processor.df

In [85]:
import pandas as pd
from data_ingestion import create_db_engine, query_data, read_from_web_CSV
import logging

class FieldDataProcessor:
    def __init__(self, logging_level="INFO"):  # When we instantiate this class, we can optionally specify what logs we want to see
        # initialising class with attributes we need. Refer to the code above to understand how each attribute relates to the code
        self.db_path = 'sqlite:///Maji_Ndogo_farm_survey_small.db'
        self.sql_query = """
            SELECT *
            FROM geographic_features
            LEFT JOIN weather_features USING (Field_ID)
            LEFT JOIN soil_and_crop_features USING (Field_ID)
            LEFT JOIN farm_management_features USING (Field_ID)
        """
        self.columns_to_rename = {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'}
        self.values_to_rename = {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
        self.weather_map_data = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"
        
        self.initialize_logging(logging_level)
        
        # We create empty objects to store the DataFrame and engine in
        self.df = None
        self.engine = None
    
    # This method enables logging in the class
    def initialize_logging(self, logging_level):
        """
        Sets up Logging for this instance of FieldDataProcessor
        """
        logger_name = __name__ + ".FieldDataProcessor"
        self.logger = logging.getLogger(logger_name)
        self.logger.propagate = False  # Prevents log messages from being propagated to the root logger
        
        # Set logging level
        if logging_level.upper() == "DEBUG":
            log_level = logging.DEBUG
        elif logging_level.upper() == "INFO":
            log_level = logging.INFO
        elif logging_level.upper() == "NONE":  # Option to disable logging
            self.logger.disabled = True
            return
        else:
            log_level = logging.INFO  # Default to INFO
        
        self.logger.setLevel(log_level)
        
        # Only add handler if not already added to avoid duplicate messages
        if not self.logger.handlers:
            ch = logging.StreamHandler()  # Create console handler
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            ch.setFormatter(formatter)
            self.logger.addHandler(ch)
        
        # Use self.logger.info(), self.logger.debug(), etc.
    
    # DataFrame methods
    def ingest_sql_data(self):
        self.engine = create_db_engine(self.db_path)
        self.df = query_data(self.engine, self.sql_query)
        self.logger.info("Successfully loaded data.")
        return self.df
            
    def rename_columns(self):
        # Annual_yield and Crop_type must be swapped
        pass
    
    def apply_corrections(self):
        # Merge the weather station data to the main DataFrame
        pass
    
    def weather_station_mapping(self):
        # Merge the weather station data to the main DataFrame
        pass
    
    def process(self):
        # This process calls the correct methods and applies the changes, step by step. This is the method we will call, and it will call the other methods in order
        weather_map_df = self.weather_station_mapping()
        self.df = self.ingest_sql_data()
        self.df = self.rename_columns()
        self.df = self.apply_corrections()
        self.df = self.df.merge(weather_map_df, on='Field_ID', how='left')
        self.df = self.df.drop(columns="Unnamed: 0")

In [86]:
field_processor = FieldDataProcessor()
field_processor.ingest_sql_data()
field_df = field_processor.df
print(field_df.shape)

2026-07-01 14:08:02,126 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 14:08:05,492 - data_ingestion - INFO - Query executed successfully.
2026-07-01 14:08:05,503 - __main__.FieldDataProcessor - INFO - Successfully loaded data.


(5654, 18)


In [89]:
import pandas as pd
from data_ingestion import create_db_engine, query_data, read_from_web_CSV
import logging

class FieldDataProcessor:
    def __init__(self, logging_level="INFO"):  # When we instantiate this class, we can optionally specify what logs we want to see
        # initialising class with attributes we need. Refer to the code above to understand how each attribute relates to the code
        self.db_path = 'sqlite:///Maji_Ndogo_farm_survey_small.db'
        self.sql_query = """
            SELECT *
            FROM geographic_features
            LEFT JOIN weather_features USING (Field_ID)
            LEFT JOIN soil_and_crop_features USING (Field_ID)
            LEFT JOIN farm_management_features USING (Field_ID)
        """
        self.columns_to_rename = {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'}
        self.values_to_rename = {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
        self.weather_map_data = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"
        
        self.initialize_logging(logging_level)
        
        # We create empty objects to store the DataFrame and engine in
        self.df = None
        self.engine = None
    
    # This method enables logging in the class
    def initialize_logging(self, logging_level):
        """
        Sets up Logging for this instance of FieldDataProcessor
        """
        logger_name = __name__ + ".FieldDataProcessor"
        self.logger = logging.getLogger(logger_name)
        self.logger.propagate = False  # Prevents log messages from being propagated to the root logger
        
        # Set logging level
        if logging_level.upper() == "DEBUG":
            log_level = logging.DEBUG
        elif logging_level.upper() == "INFO":
            log_level = logging.INFO
        elif logging_level.upper() == "NONE":  # Option to disable logging
            self.logger.disabled = True
            return
        else:
            log_level = logging.INFO  # Default to INFO
        
        self.logger.setLevel(log_level)
        
        # Only add handler if not already added to avoid duplicate messages
        if not self.logger.handlers:
            ch = logging.StreamHandler()  # Create console handler
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            ch.setFormatter(formatter)
            self.logger.addHandler(ch)
        
        # Use self.logger.info(), self.logger.debug(), etc.
    
    # DataFrame methods
    def ingest_sql_data(self):
        self.engine = create_db_engine(self.db_path)
        self.df = query_data(self.engine, self.sql_query)
        self.logger.info("Successfully loaded data.")
        return self.df
            
    def rename_columns(self):                
        # Extract the columns to rename from the configuration
        column1, column2 = list(self.columns_to_rename.keys())[0], list(self.columns_to_rename.values())[0]
        
        # Temporarily rename one of the columns to avoid a naming conflict
        temp_name = "__temp_name_for_swap__"
        while temp_name in self.df.columns:
            temp_name += "_"
        
        self.logger.info(f"Swapped columns: {column1} with {column2}")
        
        # Perform the swap
        self.df = self.df.rename(columns={column1: temp_name, column2: column1})
        self.df = self.df.rename(columns={temp_name: column2})
    
    def apply_corrections(self):
        # Merge the weather station data to the main DataFrame
        pass
    
    def weather_station_mapping(self):
        # Merge the weather station data to the main DataFrame
        pass
    
    def process(self):
        # This process calls the correct methods and applies the changes, step by step. This is the method we will call, and it will call the other methods in order
        weather_map_df = self.weather_station_mapping()
        self.df = self.ingest_sql_data()
        self.df = self.rename_columns()
        self.df = self.apply_corrections()
        self.df = self.df.merge(weather_map_df, on='Field_ID', how='left')
        self.df = self.df.drop(columns="Unnamed: 0")

In [90]:
field_processor = FieldDataProcessor()
field_processor.ingest_sql_data()
field_processor.rename_columns()
field_df = field_processor.df
field_df['Annual_yield'].head(3)

2026-07-01 14:19:32,874 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 14:19:33,350 - data_ingestion - INFO - Query executed successfully.
2026-07-01 14:19:33,350 - __main__.FieldDataProcessor - INFO - Successfully loaded data.
2026-07-01 14:19:33,454 - __main__.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type


0    0.751354
1    1.069865
2    2.208801
Name: Annual_yield, dtype: float64

In [94]:
import pandas as pd
from data_ingestion import create_db_engine, query_data, read_from_web_CSV
import logging

class FieldDataProcessor:
    def __init__(self, logging_level="INFO"):  # When we instantiate this class, we can optionally specify what logs we want to see
        # initialising class with attributes we need. Refer to the code above to understand how each attribute relates to the code
        self.db_path = 'sqlite:///Maji_Ndogo_farm_survey_small.db'
        self.sql_query = """
            SELECT *
            FROM geographic_features
            LEFT JOIN weather_features USING (Field_ID)
            LEFT JOIN soil_and_crop_features USING (Field_ID)
            LEFT JOIN farm_management_features USING (Field_ID)
        """
        self.columns_to_rename = {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'}
        self.values_to_rename = {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
        self.weather_map_data = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"
        
        self.initialize_logging(logging_level)
        
        # We create empty objects to store the DataFrame and engine in
        self.df = None
        self.engine = None
    
    # This method enables logging in the class
    def initialize_logging(self, logging_level):
        """
        Sets up Logging for this instance of FieldDataProcessor
        """
        logger_name = __name__ + ".FieldDataProcessor"
        self.logger = logging.getLogger(logger_name)
        self.logger.propagate = False  # Prevents log messages from being propagated to the root logger
        
        # Set logging level
        if logging_level.upper() == "DEBUG":
            log_level = logging.DEBUG
        elif logging_level.upper() == "INFO":
            log_level = logging.INFO
        elif logging_level.upper() == "NONE":  # Option to disable logging
            self.logger.disabled = True
            return
        else:
            log_level = logging.INFO  # Default to INFO
        
        self.logger.setLevel(log_level)
        
        # Only add handler if not already added to avoid duplicate messages
        if not self.logger.handlers:
            ch = logging.StreamHandler()  # Create console handler
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            ch.setFormatter(formatter)
            self.logger.addHandler(ch)
        
        # Use self.logger.info(), self.logger.debug(), etc.
    
    # DataFrame methods
    def ingest_sql_data(self):
        self.engine = create_db_engine(self.db_path)
        self.df = query_data(self.engine, self.sql_query)
        self.logger.info("Successfully loaded data.")
        return self.df
            
    def rename_columns(self):                
        # Extract the columns to rename from the configuration
        column1, column2 = list(self.columns_to_rename.keys())[0], list(self.columns_to_rename.values())[0]
        
        # Temporarily rename one of the columns to avoid a naming conflict
        temp_name = "__temp_name_for_swap__"
        while temp_name in self.df.columns:
            temp_name += "_"
        
        self.logger.info(f"Swapped columns: {column1} with {column2}")
        
        # Perform the swap
        self.df = self.df.rename(columns={column1: temp_name, column2: column1})
        self.df = self.df.rename(columns={temp_name: column2})
    
    def apply_corrections(self, column_name='Crop_type', abs_column='Elevation'):
        self.df[abs_column] = self.df[abs_column].abs()
        self.df[column_name] = self.df[column_name].apply(lambda crop: self.values_to_rename.get(crop, crop))
    
    def weather_station_mapping(self):
        # Merge the weather station data to the main DataFrame
        pass
    
    def process(self):
        # This process calls the correct methods and applies the changes, step by step. This is the method we will call, and it will call the other methods in order
        weather_map_df = self.weather_station_mapping()
        self.df = self.ingest_sql_data()
        self.df = self.rename_columns()
        self.df = self.apply_corrections()
        self.df = self.df.merge(weather_map_df, on='Field_ID', how='left')
        self.df = self.df.drop(columns="Unnamed: 0")

In [97]:
field_processor = FieldDataProcessor()
field_processor.ingest_sql_data()
field_processor.rename_columns()
field_processor.apply_corrections()

field_df = field_processor.df
field_df.query("Crop_type in ['cassava1', 'wheatn', 'teaa']")

2026-07-01 14:29:00,057 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 14:29:00,270 - data_ingestion - INFO - Query executed successfully.
2026-07-01 14:29:00,273 - __main__.FieldDataProcessor - INFO - Successfully loaded data.
2026-07-01 14:29:00,273 - __main__.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type


,Field_ID,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Annual_yield,Crop_type,Standard_yield


In [110]:
import pandas as pd
from data_ingestion import create_db_engine, query_data, read_from_web_CSV
import logging

class FieldDataProcessor:
    def __init__(self, logging_level="INFO"):  # When we instantiate this class, we can optionally specify what logs we want to see
        # initialising class with attributes we need. Refer to the code above to understand how each attribute relates to the code
        self.db_path = 'sqlite:///Maji_Ndogo_farm_survey_small.db'
        self.sql_query = """
            SELECT *
            FROM geographic_features
            LEFT JOIN weather_features USING (Field_ID)
            LEFT JOIN soil_and_crop_features USING (Field_ID)
            LEFT JOIN farm_management_features USING (Field_ID)
        """
        self.columns_to_rename = {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'}
        self.values_to_rename = {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
        self.weather_map_data = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"
        
        self.initialize_logging(logging_level)
        
        # We create empty objects to store the DataFrame and engine in
        self.df = None
        self.engine = None
    
    # This method enables logging in the class
    def initialize_logging(self, logging_level):
        """
        Sets up Logging for this instance of FieldDataProcessor
        """
        logger_name = __name__ + ".FieldDataProcessor"
        self.logger = logging.getLogger(logger_name)
        self.logger.propagate = False  # Prevents log messages from being propagated to the root logger
        
        # Set logging level
        if logging_level.upper() == "DEBUG":
            log_level = logging.DEBUG
        elif logging_level.upper() == "INFO":
            log_level = logging.INFO
        elif logging_level.upper() == "NONE":  # Option to disable logging
            self.logger.disabled = True
            return
        else:
            log_level = logging.INFO  # Default to INFO
        
        self.logger.setLevel(log_level)
        
        # Only add handler if not already added to avoid duplicate messages
        if not self.logger.handlers:
            ch = logging.StreamHandler()  # Create console handler
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            ch.setFormatter(formatter)
            self.logger.addHandler(ch)
        
        # Use self.logger.info(), self.logger.debug(), etc.
    
    # DataFrame methods
    def ingest_sql_data(self):
        self.engine = create_db_engine(self.db_path)
        self.df = query_data(self.engine, self.sql_query)
        self.logger.info("Successfully loaded data.")
        return self.df
            
    def rename_columns(self):                
        # Extract the columns to rename from the configuration
        column1, column2 = list(self.columns_to_rename.keys())[0], list(self.columns_to_rename.values())[0]
        
        # Temporarily rename one of the columns to avoid a naming conflict
        temp_name = "__temp_name_for_swap__"
        while temp_name in self.df.columns:
            temp_name += "_"
        
        self.logger.info(f"Swapped columns: {column1} with {column2}")
        
        # Perform the swap
        self.df = self.df.rename(columns={column1: temp_name, column2: column1})
        self.df = self.df.rename(columns={temp_name: column2})
    
    def apply_corrections(self, column_name='Crop_type', abs_column='Elevation'):
        self.df[abs_column] = self.df[abs_column].abs()
        self.df[column_name] = self.df[column_name].apply(lambda crop: self.values_to_rename.get(crop, crop))
    
    def weather_station_mapping(self):
        weather_map_df = read_from_web_CSV(self.weather_map_data)
        self.df = self.df.merge(weather_map_df, on='Field_ID', how='left')
        return self.df
    
    def process(self):
        # This process calls the correct methods and applies the changes, step by step. This is the method we will call, and it will call the other methods in order
#         weather_map_df = self.weather_station_mapping()
#         self.df = self.ingest_sql_data()
#         self.df = self.rename_columns()
#         self.df = self.apply_corrections()
#         self.df = self.df.merge(weather_map_df, on='Field_ID', how='left')
#         self.df = self.df.drop(columns="Unnamed: 0")
        pass

In [111]:
field_processor = FieldDataProcessor()
field_processor.ingest_sql_data()
field_processor.rename_columns()
field_processor.apply_corrections()
field_processor.weather_station_mapping()

field_df = field_processor.df
field_df['Weather_station'].unique()

2026-07-01 14:44:21,166 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 14:44:21,409 - data_ingestion - INFO - Query executed successfully.
2026-07-01 14:44:21,411 - __main__.FieldDataProcessor - INFO - Successfully loaded data.
2026-07-01 14:44:21,415 - __main__.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type
2026-07-01 14:44:21,947 - data_ingestion - INFO - CVS file read successfully from the web.


array([4, 0, 1, 2, 3], dtype=int64)

In [118]:
import pandas as pd
from data_ingestion import create_db_engine, query_data, read_from_web_CSV
import logging

class FieldDataProcessor:
    def __init__(self, logging_level="INFO"):  # When we instantiate this class, we can optionally specify what logs we want to see
        # initialising class with attributes we need. Refer to the code above to understand how each attribute relates to the code
        self.db_path = 'sqlite:///Maji_Ndogo_farm_survey_small.db'
        self.sql_query = """
            SELECT *
            FROM geographic_features
            LEFT JOIN weather_features USING (Field_ID)
            LEFT JOIN soil_and_crop_features USING (Field_ID)
            LEFT JOIN farm_management_features USING (Field_ID)
        """
        self.columns_to_rename = {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'}
        self.values_to_rename = {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
        self.weather_map_data = "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv"
        
        self.initialize_logging(logging_level)
        
        # We create empty objects to store the DataFrame and engine in
        self.df = None
        self.engine = None
    
    # This method enables logging in the class
    def initialize_logging(self, logging_level):
        """
        Sets up Logging for this instance of FieldDataProcessor
        """
        logger_name = __name__ + ".FieldDataProcessor"
        self.logger = logging.getLogger(logger_name)
        self.logger.propagate = False  # Prevents log messages from being propagated to the root logger
        
        # Set logging level
        if logging_level.upper() == "DEBUG":
            log_level = logging.DEBUG
        elif logging_level.upper() == "INFO":
            log_level = logging.INFO
        elif logging_level.upper() == "NONE":  # Option to disable logging
            self.logger.disabled = True
            return
        else:
            log_level = logging.INFO  # Default to INFO
        
        self.logger.setLevel(log_level)
        
        # Only add handler if not already added to avoid duplicate messages
        if not self.logger.handlers:
            ch = logging.StreamHandler()  # Create console handler
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            ch.setFormatter(formatter)
            self.logger.addHandler(ch)
        
        # Use self.logger.info(), self.logger.debug(), etc.
    
    # DataFrame methods
    def ingest_sql_data(self):
        self.engine = create_db_engine(self.db_path)
        self.df = query_data(self.engine, self.sql_query)
        self.logger.info("Successfully loaded data.")
        return self.df
            
    def rename_columns(self):                
        # Extract the columns to rename from the configuration
        column1, column2 = list(self.columns_to_rename.keys())[0], list(self.columns_to_rename.values())[0]
        
        # Temporarily rename one of the columns to avoid a naming conflict
        temp_name = "__temp_name_for_swap__"
        while temp_name in self.df.columns:
            temp_name += "_"
        
        self.logger.info(f"Swapped columns: {column1} with {column2}")
        
        # Perform the swap
        self.df = self.df.rename(columns={column1: temp_name, column2: column1})
        self.df = self.df.rename(columns={temp_name: column2})
        return self.df
    
    def apply_corrections(self, column_name='Crop_type', abs_column='Elevation'):
        self.df[abs_column] = self.df[abs_column].abs()
        self.df[column_name] = self.df[column_name].apply(lambda crop: self.values_to_rename.get(crop, crop))
        return self.df
    
    def weather_station_mapping(self):
        weather_map_df = read_from_web_CSV(self.weather_map_data)
        self.df = self.df.merge(weather_map_df, on='Field_ID', how='left')
        return self.df
    
    def process(self):
        # This process calls the correct methods and applies the changes, step by step. This is the method we will call, and it will call the other methods in order
        self.df = self.ingest_sql_data()
        self.df = self.rename_columns()
        self.df = self.apply_corrections()
        self.df = self.weather_station_mapping()
        return self.df

In [119]:
field_processor = FieldDataProcessor()
field_processor.process()

field_df = field_processor.df
field_df['Weather_station'].unique()

2026-07-01 14:49:55,616 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 14:49:55,851 - data_ingestion - INFO - Query executed successfully.
2026-07-01 14:49:55,855 - __main__.FieldDataProcessor - INFO - Successfully loaded data.
2026-07-01 14:49:55,860 - __main__.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type
2026-07-01 14:49:56,614 - data_ingestion - INFO - CVS file read successfully from the web.


array([4, 0, 1, 2, 3], dtype=int64)

In [121]:
config_params = {
    "sql_query": """
        SELECT *
        FROM geographic_features
        LEFT JOIN weather_features using (Field_ID)
        LEFT JOIN soil_and_crop_features USING (Field_ID)
        LEFT JOIN farm_management_features USING (Field_ID)
    """,
    "db_path": "sqlite:///Maji_Ndogo_farm_survey_small.db",
    "columns_to_rename": {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'},
    "values_to_rename": {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'},
    "weather_csv_path": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv",
    "weather_mapping_csv": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv",
}

In [126]:
import pandas as pd
from data_ingestion import create_db_engine, query_data, read_from_web_CSV
import logging

class FieldDataProcessor:
    def __init__(self, config_params, logging_level="INFO"):  # When we instantiate this class, we can optionally specify what logs we want to see
        # initialising class with attributes we need. Refer to the code above to understand how each attribute relates to the code
        self.db_path = config_params['db_path']
        self.sql_query = config_params['sql_query']
        self.columns_to_rename = config_params['columns_to_rename']
        self.values_to_rename = config_params['values_to_rename']
        self.weather_map_data = config_params['weather_mapping_csv']
        
        self.initialize_logging(logging_level)
        
        # We create empty objects to store the DataFrame and engine in
        self.df = None
        self.engine = None
    
    # This method enables logging in the class
    def initialize_logging(self, logging_level):
        """
        Sets up Logging for this instance of FieldDataProcessor
        """
        logger_name = __name__ + ".FieldDataProcessor"
        self.logger = logging.getLogger(logger_name)
        self.logger.propagate = False  # Prevents log messages from being propagated to the root logger
        
        # Set logging level
        if logging_level.upper() == "DEBUG":
            log_level = logging.DEBUG
        elif logging_level.upper() == "INFO":
            log_level = logging.INFO
        elif logging_level.upper() == "NONE":  # Option to disable logging
            self.logger.disabled = True
            return
        else:
            log_level = logging.INFO  # Default to INFO
        
        self.logger.setLevel(log_level)
        
        # Only add handler if not already added to avoid duplicate messages
        if not self.logger.handlers:
            ch = logging.StreamHandler()  # Create console handler
            formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            ch.setFormatter(formatter)
            self.logger.addHandler(ch)
        
        # Use self.logger.info(), self.logger.debug(), etc.
    
    # DataFrame methods
    def ingest_sql_data(self):
        self.engine = create_db_engine(self.db_path)
        self.df = query_data(self.engine, self.sql_query)
        self.logger.info("Successfully loaded data.")
        return self.df
            
    def rename_columns(self):                
        # Extract the columns to rename from the configuration
        column1, column2 = list(self.columns_to_rename.keys())[0], list(self.columns_to_rename.values())[0]
        
        # Temporarily rename one of the columns to avoid a naming conflict
        temp_name = "__temp_name_for_swap__"
        while temp_name in self.df.columns:
            temp_name += "_"
        
        self.logger.info(f"Swapped columns: {column1} with {column2}")
        
        # Perform the swap
        self.df = self.df.rename(columns={column1: temp_name, column2: column1})
        self.df = self.df.rename(columns={temp_name: column2})
        return self.df
    
    def apply_corrections(self, column_name='Crop_type', abs_column='Elevation'):
        self.df[abs_column] = self.df[abs_column].abs()
        self.df[column_name] = self.df[column_name].apply(lambda crop: self.values_to_rename.get(crop, crop))
        return self.df
    
    def weather_station_mapping(self):
        weather_map_df = read_from_web_CSV(self.weather_map_data)
        self.df = self.df.merge(weather_map_df, on='Field_ID', how='left')
        return self.df
    
    def process(self):
        # This process calls the correct methods and applies the changes, step by step. This is the method we will call, and it will call the other methods in order
        self.df = self.ingest_sql_data()
        self.df = self.rename_columns()
        self.df = self.apply_corrections()
        self.df = self.weather_station_mapping()
        return self.df

In [125]:
field_processor = FieldDataProcessor(config_params)
field_processor.process()

field_df = field_processor.df
field_df['Weather_station'].unique()

2026-07-01 15:04:57,827 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 15:04:57,994 - data_ingestion - INFO - Query executed successfully.
2026-07-01 15:04:57,994 - __main__.FieldDataProcessor - INFO - Successfully loaded data.
2026-07-01 15:04:57,994 - __main__.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type
2026-07-01 15:04:58,724 - data_ingestion - INFO - CVS file read successfully from the web.


array([4, 0, 1, 2, 3], dtype=int64)

In [1]:
import re
import numpy as np
import pandas as pd
from field_data_processor import FieldDataProcessor
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

config_params = {
    "sql_query": """
        SELECT *
        FROM geographic_features
        LEFT JOIN weather_features using (Field_ID)
        LEFT JOIN soil_and_crop_features USING (Field_ID)
        LEFT JOIN farm_management_features USING (Field_ID)
    """,
    "db_path": "sqlite:///Maji_Ndogo_farm_survey_small.db",
    "columns_to_rename": {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'},
    "values_to_rename": {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'},
    "weather_csv_path": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv",
    "weather_mapping_csv": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv",
}

field_processor = FieldDataProcessor(config_params)
field_processor.process()
field_df = field_processor.df

field_df['Weather_station'].unique()

2026-07-01 16:57:25,127 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 16:57:25,300 - data_ingestion - INFO - Query executed successfully.
2026-07-01 16:57:25,301 - field_data_processor.FieldDataProcessor - INFO - Successfully loaded data.
2026-07-01 16:57:25,303 - field_data_processor.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type
2026-07-01 16:57:26,017 - data_ingestion - INFO - CVS file read successfully from the web.


array([4, 0, 1, 2, 3], dtype=int64)

In [4]:
patterns = {
    'Rainfall': r'(\d+(\.\d+)?)\s?mm',
    'Temperature': r'(\d+(\.\d+)?)\s?C',
    'Pollution_level': r'=\s*(-?\d+(\.\d+)?)|Pollution at \s*(-?\d+(\.\d+)?)',
}

config_params = {
    "sql_query": """
        SELECT *
        FROM geographic_features
        LEFT JOIN weather_features using (Field_ID)
        LEFT JOIN soil_and_crop_features USING (Field_ID)
        LEFT JOIN farm_management_features USING (Field_ID)
    """,
    "db_path": "sqlite:///Maji_Ndogo_farm_survey_small.db",
    "columns_to_rename": {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'},
    "values_to_rename": {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'},
    "weather_csv_path": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv",
    "weather_mapping_csv": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv",
    'regex_patterns': patterns
}

In [35]:
# import re
# import numpy as np
# import pandas as pd
# import logging
# from data_ingestion import read_from_web_CSV

# class WeatherDataProcessor:
#     def __init__(self, config_params, logging_level="INFO"):
#         self.weather_station_data = config_params['weather_csv_path']
#         self.patterns = config_params['regex_patterns']
#         self.weather_df = None
#         self.initialize_logging(logging_level)
    
#     def initialize_logging(self, logging_level):
#         logger_name = __name__ + ".WeatherDataProcessor"
#         self.logger = logging.getLogger(logger_name)
#         self.logger.propagate = False
        
#         if logging_level.upper() == "DEBUG":
#             log_level = logging.DEBUG
#         elif logging_level.upper() == "INFO":
#             log_level = logging.INFO
#         elif logging_level.upper() == "NONE":
#             self.logger.disabled = True
#         else:
#             log_level = logging.INFO
        
#         self.logger.setLevel(log_level)
        
#         if not self.logger.handlers:
#             ch = logging.StreamHandler()
#             formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
#             ch.setFormatter(formatter)
#             self.logger.addHandler(ch)
    
#     def weather_station_mapping(self):
#         self.weather_df = read_from_web_CSV(self.weather_station_data)
#         self.logger.info("Successfully loaded weather station data from the web")
#         # Here, you can apply any initial transformations to self.weather_df if necessary
        
#     def extract_measurement(Self, message):
#         for key, pattern in self.patterns.items():
#             match = re.search(pattern, message)
#             if match:
#                 self.logger.debug(f"Measurement extracted: {key}")
#                 return key, float(next((x for x in match.groups() if x is not None)))
#             self.logger.debug("No measurement match found.")
#             return None, None
    
#     def process_message(self):
#         if self.weather_df is not None:
#             result = self.weather_df['Message'],apply(self.extract_measurement)
#             self.weather_df['Measurement'], self.weather_df['Value'] = zip(*result)
#             self.logger.info("Messages processed and measurements extracted.")
#         else:
#             self.logger.warning("weather_df is not initialized, skipping message processing.")
#         return self.weather_df
    
#     def calculate_means(self):
#         if self.weather_df is not None:
#             means = self.weather_df.groupby(by=['Weather_station_ID', 'Measurement'])['Value'].mean()
#             self.logger.info("Mean values calculated.")
#             return means.unstack()
#         else:
#             self.logger.warning("weather_df is not initialized, cannot calculate means.")
#             return None
    
#     def process(self):
#         self.weather_station_mapping()
#         self.process_message()
#         self.logger.ingo("Data processing completed.")

In [5]:
import re
import numpy as np
import pandas as pd
from weather_data_processor import WeatherDataProcessor
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')



config_params = {
    "sql_query": """
        SELECT *
        FROM geographic_features
        LEFT JOIN weather_features using (Field_ID)
        LEFT JOIN soil_and_crop_features USING (Field_ID)
        LEFT JOIN farm_management_features USING (Field_ID)
    """,
    "db_path": "sqlite:///Maji_Ndogo_farm_survey_small.db",
    "columns_to_rename": {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'},
    "values_to_rename": {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'},
    "weather_csv_path": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv",
    "weather_mapping_csv": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv",
    'regex_patterns': patterns
}

# filed_processor = FieldDataProcessor(config_params)
# field_processor.process()
# field_df = field_processor.df

weather_processor = WeatherDataProcessor(config_params)
weather_processor.process()
weather_df = weather_processor.weather_df

weather_df['Measurement'].unique()

2026-07-01 16:58:56,611 - data_ingestion - INFO - CVS file read successfully from the web.
2026-07-01 16:58:56,611 - weather_data_processor.WeatherDataProcessor - INFO - Successfully loaded weather station data from the web
2026-07-01 16:58:56,652 - weather_data_processor.WeatherDataProcessor - INFO - Messages processed and measurements extracted.
2026-07-01 16:58:56,653 - weather_data_processor.WeatherDataProcessor - INFO - Data processing completed.


array(['Temperature', 'Pollution_level', 'Rainfall'], dtype=object)

In [7]:
import re
import numpy as np
import pandas as pd
from field_data_processor import FieldDataProcessor
from weather_data_processor import WeatherDataProcessor
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

config_params = {
    "sql_query": """
        SELECT *
        FROM geographic_features
        LEFT JOIN weather_features using (Field_ID)
        LEFT JOIN soil_and_crop_features USING (Field_ID)
        LEFT JOIN farm_management_features USING (Field_ID)
    """,
    "db_path": "sqlite:///Maji_Ndogo_farm_survey_small.db",
    "columns_to_rename": {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'},
    "values_to_rename": {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'},
    "weather_csv_path": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv",
    "weather_mapping_csv": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv",
    'regex_patterns': patterns
}

field_processor = FieldDataProcessor(config_params)
field_processor.process()
field_df = field_processor.df

weather_processor = WeatherDataProcessor(config_params)
weather_processor.process()
weather_df = weather_processor.weather_df

2026-07-01 17:01:48,170 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 17:01:48,288 - data_ingestion - INFO - Query executed successfully.
2026-07-01 17:01:48,288 - field_data_processor.FieldDataProcessor - INFO - Successfully loaded data.
2026-07-01 17:01:48,290 - field_data_processor.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type
2026-07-01 17:01:48,737 - data_ingestion - INFO - CVS file read successfully from the web.
2026-07-01 17:01:49,284 - data_ingestion - INFO - CVS file read successfully from the web.
2026-07-01 17:01:49,285 - weather_data_processor.WeatherDataProcessor - INFO - Successfully loaded weather station data from the web
2026-07-01 17:01:49,318 - weather_data_processor.WeatherDataProcessor - INFO - Messages processed and measurements extracted.
2026-07-01 17:01:49,318 - weather_data_processor.WeatherDataProcessor - INFO - Data processing completed.


In [9]:
# !pip install pytest

weather_df.to_csv('sampled_weather_df.csv', index=False)
field_df.to_csv('sampled_field_df.csv', index=False)

!pytest validate_data.py -v

import os
weather_csv_path = 'sampled_weather_df.csv'
field_csv_path = 'sampled_field_df.csv'

if os.path.exists(weather_csv_path):
    os.remove(weather_csv_path)
    print(f"Deleted {weather_csv_path}")
else:
    print(f"{weather_csv_path} does not exist.")

if os.path.exists(field_csv_path):
    os.remove(field_csv_path)
    print(f"Deleted {field_csv_path}")
else:
    print(f"{field_csv_path} does not exist.")

============================= test session starts =============================
platform win32 -- Python 3.7.1, pytest-7.1.2, pluggy-1.0.0 -- D:\Users\steph\anaconda3\envs\interactive_visuals_ALX\python.exe
cachedir: .pytest_cache
rootdir: D:\ALX\Data Science\Python Module\Integrated Project Validating Our Data 23.0
collecting ... collected 0 items

============================ no tests ran in 0.00s ============================
Deleted sampled_weather_df.csv
Deleted sampled_field_df.csv


ERROR: file or directory not found: validate_data.py



### Validating the dataset

In [10]:
import re
import numpy as np
import pandas as pd
from field_data_processor import FieldDataProcessor
from weather_data_processor import WeatherDataProcessor
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

config_params = {
    "sql_query": """
        SELECT *
        FROM geographic_features
        LEFT JOIN weather_features using (Field_ID)
        LEFT JOIN soil_and_crop_features USING (Field_ID)
        LEFT JOIN farm_management_features USING (Field_ID)
    """,
    "db_path": "sqlite:///Maji_Ndogo_farm_survey_small.db",
    "columns_to_rename": {'Annual_yield': 'Crop_type', 'Crop_type': 'Annual_yield'},
    "values_to_rename": {'cassava1': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'},
    "weather_csv_path": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_station_data.csv",
    "weather_mapping_csv": "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_mapping.csv",
    'regex_patterns': patterns
}

field_processor = FieldDataProcessor(config_params)
field_processor.process()
field_df = field_processor.df

weather_processor = WeatherDataProcessor(config_params)
weather_processor.process()
weather_df = weather_processor.weather_df

field_df.rename(columns={'Ave_temps': 'Temperature'}, inplace=True)

2026-07-01 17:40:12,802 - data_ingestion - INFO - Database engine created successfully.
2026-07-01 17:40:15,683 - data_ingestion - INFO - Query executed successfully.
2026-07-01 17:40:15,698 - field_data_processor.FieldDataProcessor - INFO - Successfully loaded data.
2026-07-01 17:40:15,698 - field_data_processor.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type
2026-07-01 17:40:16,675 - data_ingestion - INFO - CVS file read successfully from the web.
2026-07-01 17:40:17,565 - data_ingestion - INFO - CVS file read successfully from the web.
2026-07-01 17:40:17,566 - weather_data_processor.WeatherDataProcessor - INFO - Successfully loaded weather station data from the web
2026-07-01 17:40:17,669 - weather_data_processor.WeatherDataProcessor - INFO - Messages processed and measurements extracted.
2026-07-01 17:40:17,669 - weather_data_processor.WeatherDataProcessor - INFO - Data processing completed.


### Hypothesis

In [11]:
from scipy.stats import ttest_ind
import numpy as np

# Now, the measurements_to_compare can directly use 'Temperature', 'Rainfall', and 'Pollution_Level'
measurements_to_compare = ['Temperature', 'Rainfall', 'Pollution_level']

#### Task: Create a filter_field_data function that takes in the ***field_df*** DataFrame, the ***station_id***, and ***measurement*** type, and returns a **single column** (series) of data filtered by the ***station_id**, and ***measurement***

In [22]:
def filter_field_data(df, station_id, measurement):
    return df[df['Weather_station'] == station_id][measurement]

In [23]:
# Example for station ID 0 and Temperature
station_id = 0
alpha = 0.05
measurement = 'Temperature'

# Filter the data for the specific station and measurement
field_values = filter_field_data(field_df, station_id, measurement)
field_values

1       13.35
2       13.30
8       12.80
10      13.70
14      13.35
        ...  
5627    13.30
5630    14.25
5632    11.00
5638    13.30
5642    12.85
Name: Temperature, Length: 1375, dtype: float64

In [24]:
# Example for station ID 0 and Temperature
station_id = 0
alpha = 0.05
measurement = 'Temperature'

# Filter data for the specific station and measurement
field_values = filter_field_data(field_df, station_id, measurement)
print(f"Shape: {field_values.shape}, First value: {field_values.iloc[0]}")

Shape: (1375,), First value: 13.35


#### Task: Create a data filter function that takes in the ***weather_df*** DataFrame, the ***station_id***, and ***measurement*** type, and returns a **single** column (series) of data filtered by the ***Station_id***, and ***measurement***

In [36]:
def filter_weather_data(df, station_id, measurement):
    return df.loc[(df['Weather_station_ID'] == station_id) & (df['Measurement'] == measurement), 'Value']

In [37]:
# Example for station ID 0 and Temperature
station_id = 0
alpha = 0.05
measurement = 'Temperature'

# Filter data for the specific station and measurement
weather_values = filter_weather_data(weather_df, station_id, measurement)
weather_values

0       12.82
2       14.53
29      14.28
32      12.87
67      13.13
        ...  
1804    12.77
1805    14.13
1817    13.14
1833    14.14
1834    13.61
Name: Value, Length: 100, dtype: float64

In [38]:
# Example for station ID 0 and Temperature
station_id = 0
alpha = 0.05
measurement = 'Temperature'

# Filter data for the specific station and measurement
weather_values = filter_weather_data(weather_df, station_id, measurement)
print(f"Shape: {weather_values.shape}, First value: {weather_values.iloc[0]}")

Shape: (100,), First value: 12.82


#### Task: Create a function that calculates the t-statistic and p-value. The function should accept two single columns of data and return a tuple of the t-statistic and p-value

In [40]:
def run_ttest(Column_A, Column_B):
    t_statistic, p_value = ttest_ind(Column_A, Column_B)
    return (t_statistic, p_value)

In [41]:
# Example for station ID 0 and Temperature
station_id = 0
alpha = 0.05
measurement = 'Temperature'

# Filter data for the specific station and measurement
field_values = filter_field_data(field_df, station_id, measurement)
weather_values = filter_weather_data(weather_df, station_id, measurement)

# Perform t-test
t_stat, p_val = run_ttest(field_values, weather_values)
print(f"T-stat: {t_stat:.5f}, p-value: {p_val:.5f}")

T-stat: -0.12463, p-value: 0.90083


#### Task: Replace the \ to print out the t-test result

In [42]:
def print_ttest_results(station_id, measurement, p_val, alpha):
    """
    Interprets and prints the results of a t-test based on the p-value.
    """
    if p_val < alpha:
        print(f"Significant difference in {measurement} detected at Station {station_id}, (P-Value: {p_val:.5f} < {alpha}). Null hypothesis rejected.")
    else:
        print(f"No significant difference in {measurement} detected at Station {station_id}, (P-Value: {p_val:.5f} > {alpha}). Null hypothesis not rejected.")

In [43]:
# Example for station ID 0 and Temperature
station_id = 0

measurement = 'Temperature'

# Filter data for the specific station and measurement
field_values = filter_field_data(field_df, station_id, measurement)
weather_values = filter_weather_data(weather_df, station_id, measurement)

# Perform t-test
t_stat, p_val = run_ttest(field_values, weather_values)
print_ttest_results(station_id, measurement, p_val, alpha)

No significant difference in Temperature detected at Station 0, (P-Value: 0.90083 > 0.05). Null hypothesis not rejected.


#### Task: Create a function that loops over ***measurements_to_compare*** and all ***station_id***, perform a t-test and print the results. The function should accept ***field_df***, ***weather_df***m ***list_measurements_to_compare***, ***alpha*** the value of ***alpha*** should default to a value of 0.05. Hint: Use ***print_ttest_results()***

In [88]:
def hypothesis_results(field_df, weather_df, list_measurements_to_compare, alpha = 0.05):
    station_ids = np.sort(field_df['Weather_station'].unique())
    
    for station_id in station_ids:
        for measure in list_measurements_to_compare:
            if measure:
                field_values = filter_field_data(field_df, station_id, measure)
                weather_values = filter_weather_data(weather_df, station_id, measure)

                t_stat, p_val = run_ttest(field_values, weather_values)
                print_ttest_results(station_id, measure, p_val, alpha)

In [89]:
alpha = 0.05
hypothesis_results(field_df, weather_df, measurements_to_compare, alpha)

No significant difference in Temperature detected at Station 0, (P-Value: 0.90083 > 0.05). Null hypothesis not rejected.
No significant difference in Rainfall detected at Station 0, (P-Value: 0.20242 > 0.05). Null hypothesis not rejected.
No significant difference in Pollution_level detected at Station 0, (P-Value: 0.57141 > 0.05). Null hypothesis not rejected.
No significant difference in Temperature detected at Station 1, (P-Value: 0.44064 > 0.05). Null hypothesis not rejected.
No significant difference in Rainfall detected at Station 1, (P-Value: 0.52305 > 0.05). Null hypothesis not rejected.
No significant difference in Pollution_level detected at Station 1, (P-Value: 0.24538 > 0.05). Null hypothesis not rejected.
No significant difference in Temperature detected at Station 2, (P-Value: 0.88619 > 0.05). Null hypothesis not rejected.
No significant difference in Rainfall detected at Station 2, (P-Value: 0.33897 > 0.05). Null hypothesis not rejected.
No significant difference in Poll